---
# Part C — Capstone Challenge

Build a single, complete `create_agent` for GreenPlate that combines **at least five** of the
following in one working system (your choice which five, but justify your choices in a markdown
cell before your code):

- A structured schema for orders or reservations (with at least one real constraint, like a
  `Literal` or a numeric range)
- At least two custom tools
- Long-term memory via `ToolRuntime.store` (e.g. remembering a customer's dietary preferences or
  favorite dish across sessions)
- Short-term memory via a checkpointer and `thread_id` (e.g. remembering the customer's name
  within one conversation)
- Dynamic tool gating based on some condition (membership tier, time of day, order size — your
  choice)
- A `context_schema` carrying some per-run data a tool reads (e.g. `restaurant_location`)

**Requirements:**
1. A markdown cell explaining your design choices before the code.
2. The complete, runnable code.
3. At least two `.invoke()` calls that demonstrate the system actually working — not just that
   it builds without error.
4. A short markdown reflection (3-5 sentences) on ONE trade-off or limitation of your design —
   what would break, or what would you need to add, if this went to real production use.

This is intentionally open-ended. There is no single correct architecture — the goal is
demonstrating you can combine these pieces into something coherent, not matching a hidden answer
key.


# GreenPlate is a restaurant.

#### Customers chat with an AI waiter.

#### The waiter should:

* remember previous conversations
* remember customer preferences forever
* place validated orders
* only allow Gold members to reserve priority tables

#### That is exactly what this code builds.

# Overall Architecture of the Green Plate

                    User

                      │
                      ▼

              LangGraph Agent
                      │
          ┌───────────┼────────────┐
          │           │            │
          ▼           ▼            ▼

      Middleware   Runtime     Memory

                      │
          ┌───────────┴────────────┐
          │                        │

  Short-term Memory         Long-term Memory
(Checkpointer)              (Store)

          │                        │

      Conversation         Customer Preferences

## This is long-term memory.
from langgraph.store.memory import InMemoryStore

Database

Customer
↓

* likes vegan food
* favorite dish
* allergies

This survives across different conversations (as long as the store instance exists).

This is short-term memory

from langgraph.checkpoint.memory import InMemorySaver

Conversation

User:
Hi

AI:
Hello

User:
My name is Maya

AI remembers...

# ToolRuntime

One of the most important classes.

It gives tools access to:

* context
* memory
* store
* config
* state

Instead of passing everything manually.

# Loading Environment``

In [7]:
load_dotenv()

True

In [9]:
assert os.environ.get("GROQ_API_KEY"), "Missing GROQ_API_KEY Any -- check your .env file or Colab Secrets"

# GreenPlateContext

In [10]:
# @dataclass
# class GreenPlateContext:
#   customer_id: str
#   restaurant_location: str
#   membership_tier: Literal["standard", "gold"]

#for every tool langraph injects one object

#Runtime
      │
      ▼

Context

customer_id

restaurant

membership

Every tool receives it automatically.

# Why dataclass?

In [12]:
# Instead of context["customer_id"] we write runtime.context.customer_id

# OrderItem

# quantity

# Field(ge=1,le=10)

In [ ]:
# Optional : Can be no onions

# OrderRequest

In [13]:
#Represents the whole order

Customer

↓

Name

↓

Pickup

↓

Items

↓

Address

# MENU

# Acts as a fake database.

garden bowl

↓

price

↓

tags

| Feature  | Checkpointer         | Store                    |
| -------- | -------------------- | ------------------------ |
| Purpose  | Conversation history | Persistent customer data |
| Scope    | Per `thread_id`      | Per `customer_id`        |
| Lifetime | Current conversation | Across conversations     |
| API      | `InMemorySaver`      | `InMemoryStore`          |


# Execution FLow

User

↓

Agent

↓

Middleware

↓

Allowed Tools

↓

LLM decides

↓

Tool

↓

Store / Memory

↓

Response

In [16]:
# Keytakeaways

* ToolRuntime injects context, memory, and configuration into tools.
* InMemorySaver provides short-term, thread-scoped conversation memory.
* InMemoryStore provides long-term customer preference storage.
* Pydantic models ensure only valid, structured orders reach the tools.

In [33]:
from typing import Any
from langgraph.store.memory import InMemoryStore
from langchain.tools import tool, ToolRuntime
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("GROQ_API_KEY"), "Missing GROQ_API_KEY -- check your .env file or Colab Secrets"
assert os.environ.get("OPENAI_API_KEY"), "Missing OPENAI_API_KEY -- check your .env file or Colab Secrets"


from dataclasses import dataclass
from typing import Callable, Literal , Annotated

from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
from langchain.tools import ToolRuntime, tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore


# Per-run data. It is injected when agent.invoke(...) is called.
@dataclass
class GreenPlateContext:
    customer_id: str
    restaurant_location: str
    membership_tier: Literal["standard", "gold"]


# Structured order schema with real constraints.
class OrderItem(BaseModel):
    dish: Literal[
        "Paneer Chilli",
        "Biryani",
        "Soup",
        "Mushroom Masala",
        "Russian Salad",
    ]
    quantity: int = Field(ge=1, le=10, description="Number of portions, from 1 to 10.")
    special_instructions: str | None = None


class OrderRequest(BaseModel):
    
    customer_name: str = Field(min_length=2, max_length=60)
    fulfillment: Literal["pickup", "delivery"]
    items: list[OrderItem] = Field(min_length=1, max_length=8)
    delivery_address: str | None = None


MENU = {
    "Paneer Chilli": {"price": 105.50},
    "Biryani": {"price": 150.00},
    "Soup": {"price": 75.00},
    "Mushroom Masala": {"price": 114.00},
    "Russian Salad": {"price": 80.50},
}


@tool
def view_menu(runtime: ToolRuntime[GreenPlateContext]) -> str:
    """Show GreenPlate's menu at the restaurant location for this run."""
    location = runtime.context.restaurant_location
    lines = [
        f"{dish.replace('_', ' ').title()}: ${details['price']:.2f} "
        for dish, details in MENU.items()
    ]
    return f"GreenPlate menu for {location}:\n" + "\n".join(lines)


@tool
def save_dietary_preferences(
    preferences: list[str],
    favorite_dish: str | None,
    runtime: ToolRuntime[GreenPlateContext],
) -> str:
    """Save a customer's dietary preferences and optional favorite dish for future sessions."""
    assert runtime.store is not None

    customer_id = runtime.context.customer_id
    runtime.store.put(
        ("greenplate", "customer_preferences"),
        customer_id,
        {
            "dietary_preferences": preferences,
            "favorite_dish": favorite_dish,
        },
    )
    return f"Saved preferences for customer {customer_id}."


@tool
def get_dietary_preferences(runtime: ToolRuntime[GreenPlateContext]) -> str:
    """Retrieve the customer's saved dietary preferences from long-term memory."""
    assert runtime.store is not None

    customer_id = runtime.context.customer_id
    memory = runtime.store.get(("greenplate", "customer_preferences"), customer_id)

    if memory is None:
        return "No saved dietary preferences were found for this customer."

    preferences = memory.value["dietary_preferences"]
    favorite = memory.value.get("favorite_dish") or "not set"
    return f"Dietary preferences: {', '.join(preferences)}. Favorite dish: {favorite}."


@tool
def place_order(
    order: OrderRequest,
    runtime: ToolRuntime[GreenPlateContext],
) -> str:
    """Place a validated GreenPlate food order."""
    if order.fulfillment == "delivery" and not order.delivery_address:
        return "A delivery address is required for delivery orders."

    total = sum(MENU[item.dish]["price"] * item.quantity for item in order.items)
    order_summary = ", ".join(
        f"{item.quantity}x {item.dish.replace('_', ' ')}" for item in order.items
    )
    location = runtime.context.restaurant_location

    return (
        f"Order confirmed for {order.customer_name} at {location}: {order_summary}. "
        f"Fulfillment: {order.fulfillment}. Total: ${total:.2f}."
    )



@tool
def reserve_priority_table(
    customer_name: str,
    runtime: ToolRuntime[GreenPlateContext],
    party_size: Annotated[int, Field(ge=1, le=10)] = 2,
    reservation_time: str = "19:00",
) -> str:
    """Reserve a priority table. Available only to Gold members."""
    location = runtime.context.restaurant_location
    return (
        f"Priority table reserved for {customer_name}: {party_size} guests at "
        f"{reservation_time}, GreenPlate {location}."
    )


# Dynamic tool gating: Gold members see all tools; Standard members cannot
# ask the model to call the priority reservation tool.
@wrap_model_call
def membership_tool_gate(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    tier = request.runtime.context.membership_tier

    if tier != "gold":
        allowed_tools = [
            tool for tool in request.tools if tool.name != "reserve_priority_table"
        ]
        request = request.override(tools=allowed_tools)

    return handler(request)


agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[
        view_menu,
        save_dietary_preferences,
        get_dietary_preferences,
        place_order,
        reserve_priority_table,
    ],
    system_prompt=(
        "You are GreenPlate's friendly restaurant assistant. "
        "Use tools for menu, preference, reservation, and order requests. "
        "Ask for any missing information before placing an order. "
        "Never claim a priority table was reserved unless the reservation tool succeeds."
    ),
    context_schema=GreenPlateContext,
    middleware=[membership_tool_gate],
    checkpointer=InMemorySaver(),  # short-term, thread-scoped conversation memory
    store=InMemoryStore(),         # long-term customer preferences across threads
)


# --- Example: first conversation, remembers both name in the thread and preferences in store ---
first_visit = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Hi, I'm Maya. Please remember that I am vegan and gluten-free, "
                    "and that my favorite dish is the lentil curry."
                ),
            }
        ]
    },
    config={"configurable": {"thread_id": "maya-lunch-001"}},
    context=GreenPlateContext(
        customer_id="customer-maya-123",
        restaurant_location="Indiranagar",
        membership_tier="standard",
    ),
)

print(first_visit["messages"][-1].content)


# Same thread: the checkpointer retains the previous conversation (including Maya's name).
same_thread = agent.invoke(
    {"messages": [{"role": "user", "content": "What do you remember about me?"}]},
    config={"configurable": {"thread_id": "maya-lunch-001"}},
    context=GreenPlateContext(
        customer_id="customer-maya-123",
        restaurant_location="Indiranagar",
        membership_tier="standard",
    ),
)

print(same_thread["messages"][-1].content)


# New thread: the conversation is new, but saved preferences remain available via the store.
new_session = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "I'm ordering again. What dietary preferences do I have saved?",
            }
        ]
    },
    config={"configurable": {"thread_id": "maya-dinner-002"}},
    context=GreenPlateContext(
        customer_id="customer-maya-123",
        restaurant_location="Koramangala",
        membership_tier="gold",
    ),
)

print(new_session["messages"][-1].content)

Would you like to view the menu or place an order?
I remember that you are vegan and gluten-free, and that your favorite dish is the lentil curry. I've saved these details for future reference. 

You can view our menu at any time by letting me know.
What can I help you order for lunch today?


# Checking the menu

In [34]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What food do you have today?"
            }
        ]
    },
    config={"configurable": {"thread_id": "menu-test-002"}},
    context=GreenPlateContext(
        customer_id="customer-002",
        restaurant_location="Koramangala",
        membership_tier="gold",
    ),
)



In [26]:
from langchain_core.messages import (
    HumanMessage,
    AIMessage,
    ToolMessage,
    SystemMessage,
)


def pretty_print_messages(messages):
    print("\n" + "=" * 80)
    print("📨 Conversation")
    print("=" * 80)

    for i, msg in enumerate(messages, start=1):
        print(f"\n[{i}] {'-' * 70}")

        if isinstance(msg, SystemMessage):
            print("🛡️ SYSTEM")
            print(msg.content)

        elif isinstance(msg, HumanMessage):
            print("👤 HUMAN")
            print(msg.content)

        elif isinstance(msg, AIMessage):
            print("🤖 AI")
            if msg.content:
                print(msg.content)

            if msg.tool_calls:
                print("\n🛠️ Tool Calls")
                for tool in msg.tool_calls:
                    print(f"   • Name : {tool['name']}")
                    print(f"     Args : {tool['args']}")
                    print(f"     ID   : {tool['id']}")

        elif isinstance(msg, ToolMessage):
            print("🔧 TOOL")
            print(f"Tool Name : {msg.name}")
            print(f"Tool Call : {msg.tool_call_id}")
            print("Output:")
            print(msg.content)

        else:
            print(type(msg).__name__)
            print(msg)

    print("\n" + "=" * 80)

In [35]:
pretty_print_messages(response["messages"])


📨 Conversation

[1] ----------------------------------------------------------------------
👤 HUMAN
What food do you have today?

[2] ----------------------------------------------------------------------
🤖 AI

🛠️ Tool Calls
   • Name : view_menu
     Args : {}
     ID   : tysg8f5v6

[3] ----------------------------------------------------------------------
🔧 TOOL
Tool Name : view_menu
Tool Call : tysg8f5v6
Output:
GreenPlate menu for Koramangala:
Paneer Chilli: $105.50 
Biryani: $150.00 
Soup: $75.00 
Mushroom Masala: $114.00 
Russian Salad: $80.50 

[4] ----------------------------------------------------------------------
🤖 AI
We have the following options available today. Would you like to place an order?

